In [ ]:
# =============================================================================
# TASK 3: ACTIVATION EXTRACTOR
# Hierarchical Planning in LLMs — Pass 2
#
# PURPOSE: Re-run each model with forward hooks registered. For each labeled
#          revision token, extract ±30 token windows of residual stream
#          activations, attention patterns, and layer-wise norms.
#          Save compressed numpy arrays to Drive after every problem.
#
# INPUTS:  /gdrive/MyDrive/llm_planning_study/traces/{model}/
#              {problem_id}.json          (from Task 1)
#          /gdrive/MyDrive/llm_planning_study/labels/{model}/
#              {problem_id}.json          (from Task 2)
#
# OUTPUTS: /gdrive/MyDrive/llm_planning_study/activations/{model}/
#              {problem_id}.npz           (compressed numpy arrays)
#              extraction_log.json        (progress tracker)
#          /gdrive/MyDrive/llm_planning_study/
#              activations_summary_all_models.json
#
# SPECIAL: Llama-3.1-8B-Instruct gets different handling —
#          500 random token windows instead of revision windows.
#          Set IS_BASELINE_MODEL = True for that run.
#
# RUNTIME: GPU required. One session per model.
#          Estimated: 3-5 hours per model on A100 80GB.
#
# RUN ORDER:
#   1. DeepSeek-R1-Distill-Llama-8B    IS_BASELINE_MODEL = False
#   2. Llama-3.1-8B-Instruct            IS_BASELINE_MODEL = True
#   3. DeepSeek-R1-Distill-Qwen-7B     IS_BASELINE_MODEL = False
#   4. DeepSeek-R1-Distill-Qwen-14B    IS_BASELINE_MODEL = False
#   5. DeepSeek-R1-Distill-Qwen-1.5B   IS_BASELINE_MODEL = False
# =============================================================================


# -----------------------------------------------------------------------------
# CELL 1 — Install dependencies
# -----------------------------------------------------------------------------
# !pip install transformers accelerate bitsandbytes -q


# -----------------------------------------------------------------------------
# CELL 2 — Imports and config
# -----------------------------------------------------------------------------
import gc
import json
import time
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── MODEL SELECTION ───────────────────────────────────────────────────────────
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
#MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
IS_BASELINE_MODEL = False   # True ONLY for Llama-3.1-8B-Instruct

# ── EXTRACTION CONFIG ─────────────────────────────────────────────────────────
WINDOW_TOKENS     = 30      # ±30 tokens around each revision
N_BASELINE_WINDOWS = 500    # random windows for Llama baseline
RANDOM_SEED        = 42

# ── QUANTIZATION ──────────────────────────────────────────────────────────────
USE_4BIT = "32B" in MODEL_NAME or "32b" in MODEL_NAME

# ── PATHS ─────────────────────────────────────────────────────────────────────
GDRIVE_BASE    = Path("/content/gdrive/MyDrive/llm_planning_study")
MODEL_SHORT    = MODEL_NAME.split("/")[-1]
TRACE_DIR      = GDRIVE_BASE / "traces"  / MODEL_SHORT
LABEL_DIR      = GDRIVE_BASE / "labels"  / MODEL_SHORT
ACTIV_DIR      = GDRIVE_BASE / "activations" / MODEL_SHORT
LOG_FILE       = ACTIV_DIR / "extraction_log.json"
SUMMARY_FILE   = GDRIVE_BASE / "activations_summary_all_models.json"

print(f"Model:        {MODEL_NAME}")
print(f"Baseline:     {IS_BASELINE_MODEL}")
print(f"Window:       ±{WINDOW_TOKENS} tokens")
print(f"Output dir:   {ACTIV_DIR}")


# -----------------------------------------------------------------------------
# CELL 3 — Mount Google Drive
# -----------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")
ACTIV_DIR.mkdir(parents=True, exist_ok=True)
print(f"Activation directory ready: {ACTIV_DIR}")


# -----------------------------------------------------------------------------
# CELL 4 — Load model and tokenizer
# -----------------------------------------------------------------------------
def load_model_and_tokenizer(model_name: str, use_4bit: bool):
    print(f"\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model ({'4-bit' if use_4bit else 'fp16'})...")
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )

    model.eval()

    # Detect layer count and hidden dim automatically
    config     = model.config
    n_layers   = config.num_hidden_layers
    hidden_dim = config.hidden_size
    print(f"Model loaded. Layers: {n_layers}, Hidden dim: {hidden_dim}")
    return model, tokenizer, n_layers, hidden_dim


model, tokenizer, N_LAYERS, HIDDEN_DIM = load_model_and_tokenizer(
    MODEL_NAME, USE_4BIT
)


# -----------------------------------------------------------------------------
# CELL 5 — Hook registration
# Hook into residual stream AFTER each transformer block.
# We capture output[0] which is the hidden state tensor.
# Hooks are registered once and stay active for all forward passes.
# -----------------------------------------------------------------------------
#------------------------------------------------------------------------
class ActivationStore:
    """
    Thread-safe store for activations captured by forward hooks.
    Cleared before each forward pass via .clear().
    """
    def __init__(self):
        self.data = {}

    def clear(self):
        self.data.clear()

    def __setitem__(self, key, value):
        self.data[key] = value

    def __getitem__(self, key):
        return self.data[key]

    def keys(self):
        return self.data.keys()

store = ActivationStore()


hooks = []

def register_hooks(model, store):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = model.transformer.h
    else:
        raise RuntimeError(f"Cannot find layers in {type(model)}")

    hooks = []
    for i, layer in enumerate(layers):
        def make_hook(layer_idx):
            def hook(module, input, output):
                hidden_state = output[0]
                if hidden_state.dim() == 3:
                    store[layer_idx] = hidden_state[0].detach().cpu().float()
                elif hidden_state.dim() == 2:
                    store[layer_idx] = hidden_state.detach().cpu().float()
                else:
                    print(f"WARNING layer {layer_idx}: unexpected shape {hidden_state.shape}")
            return hook
        h = layer.register_forward_hook(make_hook(i))
        hooks.append(h)

    print(f"Registered {len(hooks)} hooks.")
    return hooks

hooks = register_hooks(model, store)

# Verify immediately
store.clear()
test_input = tokenizer("test", return_tensors="pt").to(model.device)
with torch.no_grad():
    _ = model(**test_input)
print(f"Verification — keys in store: {len(store.keys())} layers")
assert len(store.keys()) == N_LAYERS, f"Expected {N_LAYERS}, got {len(store.keys())}"
print("Hooks verified OK.")
# -----------------------------------------------------------------------------
# CELL 6 — Text utilities
# -----------------------------------------------------------------------------
def clean_text(text: str) -> str:
    return (text.replace("Ġ", " ")
                .replace("Ċ", "\n")
                .replace("ĉ", "\t"))


def get_analysis_text(trace: dict) -> str:
    think = clean_text(trace.get("think_content", ""))
    clean = clean_text(trace.get("response_clean", ""))
    return think if think.strip() else clean


def word_idx_to_token_idx(
    text: str,
    word_idx: int,
    tokenizer,
    prompt_len: int
) -> int:
    """
    Convert a word index in the analysis text to a token index in the
    full model input (which includes the prompt prefix).

    Strategy:
    1. Split text into words
    2. Reconstruct text up to word_idx
    3. Tokenize that prefix to get token count
    4. Add prompt_len offset

    This is approximate but sufficient for ±30 token windows.
    """
    words  = text.split()
    prefix = " ".join(words[:word_idx])
    prefix_tokens = tokenizer(prefix, return_tensors="pt",
                              add_special_tokens=False)
    n_prefix_tokens = prefix_tokens["input_ids"].shape[1]
    return prompt_len + n_prefix_tokens


# -----------------------------------------------------------------------------
# CELL 7 — Prompt builder (identical to Task 1 to ensure same tokenization)
# -----------------------------------------------------------------------------
def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])

    if problem.get("difficulty") == "hard":
        instruction = (
            "Solve the following competition mathematics problem carefully. "
            "Show your reasoning step by step and verify your answer."
        )
    else:
        instruction = (
            "Solve the following mathematics competition problem. "
            "Show your working step by step."
        )

    if is_reasoning:
        messages = [{"role": "user",
                     "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system",
             "content": "You are an expert mathematician. Solve problems with "
                        "clear step-by-step reasoning."},
            {"role": "user",
             "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]

    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# -----------------------------------------------------------------------------
# CELL 8 — Core extraction: run model and extract windows
# -----------------------------------------------------------------------------
@torch.no_grad()
def extract_windows_for_problem(
    trace: dict,
    label: dict,
    model,
    tokenizer,
    store: ActivationStore,
    n_layers: int,
    window_tokens: int
) -> dict | None:
    """
    Extracts activation windows for a given problem based on revision labels.
    """
    pid        = trace["problem_id"]
    analysis   = get_analysis_text(trace)
    prompt     = build_prompt(trace, tokenizer, MODEL_NAME)
    inputs     = tokenizer(prompt, return_tensors="pt", truncation=False).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    # Generate full input IDs for the model run
    full_text   = prompt + analysis
    full_inputs = tokenizer(full_text, return_tensors="pt", truncation=False).to(model.device)
    seq_len     = full_inputs["input_ids"].shape[1]

    # Convert revision word indices to token indices
    revision_entries = []
    for rev in label.get("revisions", []):
        word_idx       = rev.get("revision_word_idx", 0)
        ctrl_word_idx  = rev.get("control_word_idx", 0)

        token_idx      = word_idx_to_token_idx(
            analysis, word_idx, tokenizer, prompt_len
        )
        ctrl_token_idx = word_idx_to_token_idx(
            analysis, ctrl_word_idx, tokenizer, prompt_len
        )

        # Clamp to valid range
        token_idx      = max(window_tokens, min(token_idx,      seq_len - window_tokens - 1))
        ctrl_token_idx = max(window_tokens, min(ctrl_token_idx, seq_len - window_tokens - 1))
        revision_entries.append({
            "token_idx":      token_idx,
            "ctrl_token_idx": ctrl_token_idx,   # ← this was missing
            "verdict":        rev.get("verdict", "UNKNOWN"),
            "confidence":     rev.get("confidence", 0.0),
            "word_idx":       word_idx,
        })
    if not revision_entries:
        return None # No valid revisions to extract

    # Run model forward pass to populate store
    store.clear()
    _ = model(**full_inputs, output_hidden_states=False)

    # Instead of pre-allocating the full array upfront:
    # OLD — causes OOM on large n_revisions:
    # revision_windows = np.zeros((n_revisions, n_layers, window_size, HIDDEN_DIM), dtype=np.float16)

    # NEW — build list, stack at end:
    revision_windows_list = []
    control_windows_list  = [] # Control windows are not defined in the trace, commenting out
    revision_norms_list   = []
    control_norms_list    = [] # Control windows are not defined in the trace, commenting out

    window_size = 2 * window_tokens + 1

    for r, entry in enumerate(revision_entries):
        t_rev  = entry["token_idx"]
        t_ctrl = entry["ctrl_token_idx"] # Control tokens are not defined in the trace, commenting out

        rev_win_layers  = np.zeros((n_layers, window_size, HIDDEN_DIM), dtype=np.float16)
        ctrl_win_layers = np.zeros((n_layers, window_size, HIDDEN_DIM), dtype=np.float16) # Control windows are not defined in the trace, commenting out
        rev_norms       = np.zeros(n_layers, dtype=np.float32)
        ctrl_norms      = np.zeros(n_layers, dtype=np.float32) # Control windows are not defined in the trace, commenting out

        for l in range(n_layers):
            hidden = store[l]   # (seq_len, hidden_dim)

            rev_window = hidden[t_rev - window_tokens: t_rev + window_tokens + 1]
            if rev_window.shape[0] == window_size:
                rev_win_layers[l] = rev_window.numpy().astype(np.float16)

            ctrl_window = hidden[t_ctrl - window_tokens: t_ctrl + window_tokens + 1] # Control windows are not defined in the trace, commenting out
            if ctrl_window.shape[0] == window_size:
                 ctrl_win_layers[l] = ctrl_window.numpy().astype(np.float16) # Control windows are not defined in the trace, commenting out

            if l > 0:
                prev = store[l-1]
                rev_norms[l]  = float(torch.norm(hidden[t_rev]  - prev[t_rev]).item())
                # ctrl_norms[l] = float(torch.norm(hidden[t_ctrl] - prev[t_ctrl]).item()) # Control windows are not defined in the trace, commenting out

        revision_windows_list.append(rev_win_layers)
        control_windows_list.append(ctrl_win_layers) # Control windows are not defined in the trace, commenting out
        revision_norms_list.append(rev_norms)
        control_norms_list.append(ctrl_norms) # Control windows are not defined in the trace, commenting out

    # Stack into final arrays
    revision_windows = np.stack(revision_windows_list)  # (n_rev, n_layers, 61, hidden_dim)
    control_windows  = np.stack(control_windows_list) # Control windows are not defined in the trace, commenting out
    revision_norms   = np.stack(revision_norms_list)    # (n_rev, n_layers)
    control_norms    = np.stack(control_norms_list) # Control windows are not defined in the trace, commenting out

    store.clear()
    torch.cuda.empty_cache()
    gc.collect()
    revision_meta = [
        {
            "token_idx":      entry["token_idx"],
            "ctrl_token_idx": entry["ctrl_token_idx"],
            "verdict":        entry.get("verdict", "UNKNOWN"),
            "confidence":     entry.get("confidence", 0.0),
            "word_idx":       entry.get("word_idx", 0),
        }
        for entry in revision_entries
    ]

    return {
        "problem_id":       pid,
        "model":            MODEL_SHORT,
        "difficulty":       trace.get("difficulty", "unknown"),
        "is_baseline":      False,
        "n_revisions":      len(revision_entries),
        "n_layers":         n_layers,
        "hidden_dim":       HIDDEN_DIM,
        "window_tokens":    window_tokens,
        "revision_meta":    revision_meta,
        "revision_windows": np.nan_to_num(revision_windows, nan=0.0, posinf=0.0, neginf=0.0),
        "control_windows":  np.nan_to_num(control_windows,  nan=0.0, posinf=0.0, neginf=0.0),
        "revision_norms":   revision_norms,
        "control_norms":    control_norms,
    }

# -----------------------------------------------------------------------------
# CELL 9 — Baseline extraction for Llama (random windows)
# Only runs when IS_BASELINE_MODEL = True
# -----------------------------------------------------------------------------
@torch.no_grad()
def extract_baseline_windows(
    trace: dict,
    model,
    tokenizer,
    store: ActivationStore,
    n_layers: int,
    window: int,
    n_windows: int,
    rng: random.Random
) -> dict | None:
    """
    For Llama baseline: extract N random token windows from each trace.
    No revision labels needed — just random positions in the response.
    """
    pid         = trace["problem_id"]
    analysis    = get_analysis_text(trace)
    prompt      = build_prompt(trace, tokenizer, MODEL_NAME)
    inputs      = tokenizer(prompt, return_tensors="pt",
                            truncation=False).to(model.device)
    prompt_len  = inputs["input_ids"].shape[1]

    full_text   = prompt + analysis
    full_inputs = tokenizer(full_text, return_tensors="pt",
                            truncation=False).to(model.device)
    seq_len     = full_inputs["input_ids"].shape[1]

    if seq_len <= 2 * window + prompt_len + 10:
        return None   # trace too short for any window

    store.clear()
    _ = model(**full_inputs, output_hidden_states=False)

    # Sample random positions in the response portion only
    valid_range  = range(prompt_len + window, seq_len - window - 1)
    if len(valid_range) < n_windows:
        positions = list(valid_range)
    else:
        positions = rng.sample(list(valid_range), n_windows)

    window_size = 2 * window + 1
    n_pos       = len(positions)

    baseline_windows = np.zeros(
        (n_pos, n_layers, window_size, HIDDEN_DIM), dtype=np.float16
    )
    baseline_norms   = np.zeros((n_pos, n_layers), dtype=np.float32)

    for p_idx, t in enumerate(positions):
        for l in range(n_layers):
            hidden = store[l]
            win    = hidden[t - window: t + window + 1]
            if win.shape[0] == window_size:
                baseline_windows[p_idx, l] = win.numpy().astype(np.float16)
            if l > 0:
                prev    = store[l-1][0]
                delta   = hidden[t] - prev[t]
                baseline_norms[p_idx, l] = float(torch.norm(delta).item())

    store.clear()
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "problem_id":       pid,
        "model":            MODEL_SHORT,
        "difficulty":       trace.get("difficulty", "unknown"),
        "is_baseline":      True,
        "n_windows":        n_pos,
        "n_layers":         n_layers,
        "hidden_dim":       HIDDEN_DIM,
        "window_tokens":    window,
        "positions":        positions,
        "baseline_windows": baseline_windows,   # (n_pos, n_layers, 61, hidden_dim)
        "baseline_norms":   baseline_norms,     # (n_pos, n_layers)
    }


# -----------------------------------------------------------------------------
# CELL 10 — Save and load functions
# -----------------------------------------------------------------------------
def save_activations(result: dict, activ_dir: Path):
    """
    Save activation arrays as compressed .npz and metadata as .json.
    Keeps arrays and metadata separate for efficient loading in Task 4.
    """
    pid = result["problem_id"]

    # Save arrays
    array_keys = [k for k, v in result.items() if isinstance(v, np.ndarray)]
    arrays     = {k: result[k] for k in array_keys}
    np.savez_compressed(activ_dir / f"{pid}.npz", **arrays)

    # Save metadata (everything that's not an array)
    meta = {k: v for k, v in result.items() if not isinstance(v, np.ndarray)}
    with open(activ_dir / f"{pid}_meta.json", "w") as f:
        json.dump(meta, f, indent=2)


def load_activations(pid: str, activ_dir: Path) -> tuple[dict, dict]:
    """
    Load arrays and metadata for one problem. Used in Task 4.
    """
    arrays = dict(np.load(activ_dir / f"{pid}.npz"))
    with open(activ_dir / f"{pid}_meta.json") as f:
        meta = json.load(f)
    return arrays, meta


# -----------------------------------------------------------------------------
# CELL 11 — Checkpoint log
# -----------------------------------------------------------------------------
def load_extraction_log() -> dict:
    if LOG_FILE.exists():
        with open(LOG_FILE) as f:
            log = json.load(f)
        print(f"Resuming. Done: {len(log['completed'])} | "
              f"Skipped: {len(log['skipped'])} | Failed: {len(log['failed'])}")
    else:
        log = {
            "model":       MODEL_SHORT,
            "is_baseline": IS_BASELINE_MODEL,
            "started":     datetime.now().isoformat(),
            "completed":   [],
            "skipped":     [],
            "failed":      [],
        }
        print("Starting fresh extraction run.")
    return log


def save_extraction_log(log: dict):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)


# -----------------------------------------------------------------------------
# CELL 12 — Build work queue
# For reasoning models: pair trace files with label files.
# For baseline model: use all non-truncated trace files directly.
# -----------------------------------------------------------------------------
def build_work_queue(
    trace_dir: Path,
    label_dir: Path,
    is_baseline: bool,
    completed: list
) -> list[tuple]:
    """
    Returns list of (trace, label_or_None) tuples to process.
    """
    done = set(completed)
    trace_files = [f for f in trace_dir.glob("*.json")
                   if f.name != "run_log.json"]

    queue = []
    print(len(trace_files))

    if is_baseline:
        # Baseline: all non-truncated traces, no labels needed
        for tf in trace_files:
            with open(tf) as f:
                trace = json.load(f)
            if 'problem_id' not in trace:
                print(f"Skipping trace {tf.name}: 'problem_id' key missing.")
                continue
            if trace.get("truncated"):
                continue
            if trace["problem_id"] in done:
                continue
            queue.append((trace, None))
    else:
        # Reasoning models: only traces with a valid label file
        label_files = {f.stem: f for f in label_dir.glob("*.json")
                       if f.name != "labeling_log.json"
                       and f.name != "label_summary.json"}

        for tf in trace_files:
            with open(tf) as f:
                trace = json.load(f)

            if 'problem_id' not in trace:
                print(f"Skipping trace {tf.name}: 'problem_id' key missing.")
                continue

            pid = trace["problem_id"]

            if pid in done:
                continue
            if pid not in label_files:
                continue   # no label — skipped in Task 2

            with open(label_files[pid]) as f:
                label = json.load(f)

            if label.get("skipped"):
                continue   # Task 2 decided to skip this trace

            queue.append((trace, label))

    return queue


log   = load_extraction_log()
queue = build_work_queue(
    TRACE_DIR, LABEL_DIR,
    IS_BASELINE_MODEL,
    log["completed"] + log["skipped"]
)
print(f"Work queue: {len(queue)} problems to extract")


# -----------------------------------------------------------------------------
# CELL 13 — Main extraction loop
# -----------------------------------------------------------------------------
rng = random.Random(RANDOM_SEED)

# Estimate storage before starting
if not IS_BASELINE_MODEL and queue:
    sample_label = queue[0][1]
    avg_rev_est  = sum(
        l.get("n_verified", 0) for _, l in queue if l
    ) / max(len(queue), 1)
    window_size  = 2 * WINDOW_TOKENS + 1
    est_gb       = (len(queue) * avg_rev_est * N_LAYERS
                    * window_size * HIDDEN_DIM * 2) / 1024**3
    print(f"Estimated storage: {est_gb:.1f} GB")
    print(f"  ({len(queue)} problems × {avg_rev_est:.1f} revisions × "
          f"{N_LAYERS} layers × {window_size} tokens × {HIDDEN_DIM} dims × 2 bytes)")

print(f"\nStarting extraction: {len(queue)} problems")
print(f"Model: {MODEL_NAME}")
print(f"Baseline mode: {IS_BASELINE_MODEL}")
print("=" * 65)

total_revisions_extracted = 0
for i, (trace, label) in enumerate(queue):
    pid  = trace["problem_id"]
    diff = trace.get("difficulty", "?")
    print(f"\n[{i+1}/{len(queue)}] {pid}  diff={diff}")

    t0 = time.time()
    try:
        if IS_BASELINE_MODEL:
            result = extract_baseline_windows(
                trace, model, tokenizer, store,
                N_LAYERS, WINDOW_TOKENS,
                N_BASELINE_WINDOWS // max(len(queue), 1) + 1,
                rng
            )
        else:
            result = extract_windows_for_problem(
                trace, label, model, tokenizer, store,
                N_LAYERS, WINDOW_TOKENS
            )

        if result is None:
            log["skipped"].append(pid)
            save_extraction_log(log)
            print(f"  SKIPPED: no valid windows")
            continue

        save_activations(result, ACTIV_DIR)
        log["completed"].append(pid)
        save_extraction_log(log)

        elapsed = time.time() - t0
        if IS_BASELINE_MODEL:
            n_extracted = result["n_windows"]
        else:
            n_extracted = result["n_revisions"]
            total_revisions_extracted += n_extracted

        # Estimate file size
        total_elements = sum(
            v.size for v in result.values() if isinstance(v, np.ndarray)
        )
        est_mb = total_elements * 2 / 1024**2   # float16 = 2 bytes

        print(f"  ✓ {n_extracted} windows | "
              f"~{est_mb:.1f} MB | {elapsed:.1f}s")

    except Exception as e:
        print(f"  FAILED: {e}")
        import traceback
        traceback.print_exc()
        log["failed"].append({"problem_id": pid, "error": str(e)})
        save_extraction_log(log)
        # Clear memory even on failure
        store.clear()
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "=" * 65)
print(f"Extraction complete.")
print(f"  Completed : {len(log['completed'])}")
print(f"  Skipped   : {len(log['skipped'])}")
print(f"  Failed    : {len(log['failed'])}")
if not IS_BASELINE_MODEL:
    print(f"  Total revision windows extracted: {total_revisions_extracted}")


# -----------------------------------------------------------------------------
# CELL 14 — Storage audit and cross-model summary
# Run after extraction completes. Checks actual disk usage and validates
# all .npz files are readable before moving to Task 4.
# -----------------------------------------------------------------------------
def storage_audit(activ_dir: Path, model_short: str) -> dict:
    npz_files = list(activ_dir.glob("*.npz"))
    if not npz_files:
        print("No .npz files found.")
        return {}

    total_bytes     = 0
    total_revisions = 0
    total_problems  = 0
    corrupt_files   = []
    by_diff         = defaultdict(lambda: {"problems": 0, "revisions": 0, "bytes": 0})

    for npz_path in npz_files:
        pid       = npz_path.stem
        meta_path = activ_dir / f"{pid}_meta.json"

        # Check file is readable
        try:
            arrays = np.load(npz_path)
            _ = list(arrays.keys())   # force load
        except Exception as e:
            corrupt_files.append({"pid": pid, "error": str(e)})
            continue

        file_bytes = npz_path.stat().st_size
        total_bytes += file_bytes

        # Load metadata
        if meta_path.exists():
            with open(meta_path) as f:
                meta = json.load(f)
            diff = meta.get("difficulty", "unknown")
            n_rev = meta.get("n_revisions", meta.get("n_windows", 0))
        else:
            diff  = "unknown"
            n_rev = 0

        by_diff[diff]["problems"]   += 1
        by_diff[diff]["revisions"]  += n_rev
        by_diff[diff]["bytes"]      += file_bytes
        total_revisions += n_rev
        total_problems  += 1

    total_gb = total_bytes / 1024**3

    print(f"\n{'='*65}")
    print(f"STORAGE AUDIT — {model_short}")
    print(f"{'='*65}")
    print(f"  Total problems:   {total_problems}")
    print(f"  Total revisions:  {total_revisions}")
    print(f"  Total size:       {total_gb:.2f} GB")
    print(f"  Corrupt files:    {len(corrupt_files)}")

    for diff, stats in by_diff.items():
        gb = stats["bytes"] / 1024**3
        print(f"\n  {diff.upper()}")
        print(f"    Problems:   {stats['problems']}")
        print(f"    Revisions:  {stats['revisions']}")
        print(f"    Size:       {gb:.2f} GB")

    if corrupt_files:
        print(f"\n  ⚠ CORRUPT FILES — re-extract these:")
        for c in corrupt_files:
            print(f"    {c['pid']}: {c['error']}")

    summary = {
        "model":           model_short,
        "timestamp":       datetime.now().isoformat(),
        "total_problems":  total_problems,
        "total_revisions": total_revisions,
        "total_gb":        round(total_gb, 2),
        "corrupt_files":   corrupt_files,
        "by_difficulty":   {
            d: {k: v for k, v in s.items() if k != "bytes"}
            for d, s in by_diff.items()
        }
    }

    # Save per-model audit
    audit_path = activ_dir / "storage_audit.json"
    with open(audit_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n  Audit saved: {audit_path}")

    # Append to cross-model summary
    if SUMMARY_FILE.exists():
        with open(SUMMARY_FILE) as f:
            cross = json.load(f)
    else:
        cross = {}
    cross[model_short] = summary
    with open(SUMMARY_FILE, "w") as f:
        json.dump(cross, f, indent=2)
    print(f"  Cross-model summary updated: {SUMMARY_FILE}")

    return summary


storage_audit(ACTIV_DIR, MODEL_SHORT)


Model:        deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
Baseline:     False
Window:       ±30 tokens
Output dir:   /content/gdrive/MyDrive/llm_planning_study/activations/DeepSeek-R1-Distill-Qwen-1.5B
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Activation directory ready: /content/gdrive/MyDrive/llm_planning_study/activations/DeepSeek-R1-Distill-Qwen-1.5B

Loading tokenizer...
Loading model (fp16)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded. Layers: 28, Hidden dim: 1536
Registered 28 hooks.
Verification — keys in store: 28 layers
Hooks verified OK.
Resuming. Done: 128 | Skipped: 0 | Failed: 384
301
Skipping trace pilot_analysis.json: 'problem_id' key missing.
Work queue: 0 problems to extract

Starting extraction: 0 problems
Model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
Baseline mode: False

Extraction complete.
  Completed : 128
  Skipped   : 0
  Failed    : 384
  Total revision windows extracted: 0

STORAGE AUDIT — DeepSeek-R1-Distill-Qwen-1.5B
  Total problems:   128
  Total revisions:  912
  Total size:       8.27 GB
  Corrupt files:    0

  HARD
    Problems:   84
    Revisions:  657
    Size:       5.96 GB

  MEDIUM
    Problems:   44
    Revisions:  255
    Size:       2.31 GB

  Audit saved: /content/gdrive/MyDrive/llm_planning_study/activations/DeepSeek-R1-Distill-Qwen-1.5B/storage_audit.json
  Cross-model summary updated: /content/gdrive/MyDrive/llm_planning_study/activations_summary_all_models.j

{'model': 'DeepSeek-R1-Distill-Qwen-1.5B',
 'timestamp': '2026-06-22T08:21:08.347954',
 'total_problems': 128,
 'total_revisions': 912,
 'total_gb': 8.27,
 'corrupt_files': [],
 'by_difficulty': {'hard': {'problems': 84, 'revisions': 657},
  'medium': {'problems': 44, 'revisions': 255}}}

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

LABEL_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/labels/DeepSeek-R1-Distill-Llama-8B")

files = [f for f in LABEL_DIR.glob("*.json")
         if f.name not in ("labeling_log.json", "label_summary.json")]

stats = defaultdict(lambda: defaultdict(int))

for f in files:
    with open(f) as fp:
        label = json.load(fp)
    diff   = label.get("difficulty", "unknown")
    if label.get("skipped"):
        reason = label.get("reason", "unknown")
        stats[diff][f"skipped:{reason}"] += 1
    else:
        stats[diff]["labeled"] += 1

for diff in ["hard", "medium"]:
    print(f"\n{diff.upper()}")
    for reason, count in sorted(stats[diff].items()):
        print(f"  {reason:50s}: {count}")


HARD
  labeled                                           : 86
  skipped:insufficient_revisions (0 < 1)            : 14
  skipped:no_candidates                             : 57

MEDIUM
  labeled                                           : 36
  skipped:insufficient_revisions (0 < 1)            : 7
  skipped:no_candidates                             : 50


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
import json
from pathlib import Path

TRACE_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Llama-8B")

# Find a trace that was labeled no_candidates
LABEL_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/labels/DeepSeek-R1-Distill-Llama-8B")

no_cand = []
for f in LABEL_DIR.glob("*.json"):
    if f.name in ("labeling_log.json", "label_summary.json"):
        continue
    with open(f) as fp:
        label = json.load(fp)
    if label.get("reason") == "no_candidates":
        no_cand.append(label["problem_id"])

# Check first one
pid = no_cand[0]
with open(TRACE_DIR / f"{pid}.json") as f:
    trace = json.load(f)

text = trace.get("think_content") or trace.get("response_clean", "")
print(f"Problem: {pid}")
print(f"Has Ġ artifacts: {'Ġ' in text}")
print(f"First 300 chars: {repr(text[:300])}")

In [ ]:
import json
from pathlib import Path

# Assuming pid is already defined from a previous cell, if not, choose a sample pid
# For example, using the 'pid' from cell '_ARGhh28Cheu'
if 'pid' not in locals():
    # Fallback if pid is not defined (e.g., if previous cell wasn't run)
    # You might need to adjust this to a valid problem_id from your TRACE_DIR
    sample_trace_files = list(TRACE_DIR.glob("*.json"))
    if sample_trace_files:
        sample_trace_path = sample_trace_files[0]
        pid = sample_trace_path.stem
    else:
        pid = "hard_00228" # Default to the one used in the previous cell if no files exist


print(f"Inspecting trace file for problem_id: {pid}")

# Load the trace file
trace_filepath = TRACE_DIR / f"{pid}.json"
with open(trace_filepath) as f:
    trace_data = json.load(f)

# Display the keys available in the trace_data dictionary
print("\nKeys available in the trace dictionary:")
for key in trace_data.keys():
    print(f"- {key}")

# You can also display a snippet of the data for context if needed
# print("\nFirst 2 items of the trace data:")
# display(list(trace_data.items())[:2])

In [ ]:
import json
from pathlib import Path

# Ensure TRACE_DIR is defined, as it's used from a previous cell.
# If this cell is run independently, you might need to define it here:
# TRACE_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Llama-8B")

trace_files = list(TRACE_DIR.glob("*.json"))
total_traces = len(trace_files)

print(f"Checking {total_traces} trace files for the 'product_id' key...")

missing_product_id_count = 0
for i, trace_filepath in enumerate(trace_files):
    try:
        with open(trace_filepath) as f:
            trace = json.load(f)

        if 'problem_id1' not in trace:
            print(f"Trace {i + 1} of {total_traces} (ID: {trace.get('problem_id', 'N/A')}) is missing 'product_id'.")
            missing_product_id_count += 1
        # else:
        #     print(f"Trace {i + 1} of {total_traces} (ID: {trace.get('problem_id', 'N/A')}) has 'product_id'.")

    except json.JSONDecodeError:
        print(f"Error decoding JSON for file: {trace_filepath.name}")
    except Exception as e:
        print(f"An unexpected error occurred for file {trace_filepath.name}: {e}")

if missing_product_id_count == 0:
    print("All trace files contain 'product_id' (or no files were processed).")
else:
    print(f"\nFinished check. Found 'product_id' missing in {missing_product_id_count} out of {total_traces} trace files.")



Once you see the available keys, you can update your code to use the correct key names. If a key might not always be present, you can use `.get()` with a default value, or check for its existence before accessing it, for example:

```python
value = dictionary.get('key_name', default_value)
# or
if 'key_name' in dictionary:
    value = dictionary['key_name']
```